## Image Augmentation

Cats v Dogs 로 다음처럼 모델링 하고, 학습시켜본다.

4 convolutional layers with 32, 64, 128 and 128 convolutions

train for 100 epochs

https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip

In [3]:
import os, zipfile, pathlib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [12]:
url = 'https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip'

In [13]:
zip_path = keras.utils.get_file('cats_and_dogs_filtered.zip', url, extract=False)

In [14]:
extract_root = pathlib.Path('.')

In [15]:
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_root)

In [16]:
IMG_SIZE = 150
IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3)

In [17]:
BATCH_SIZE = 20

In [18]:
SEED = 1337

In [19]:
base_dir = extract_root / 'cats_and_dogs_filtered'

In [20]:
train_dir = base_dir / 'train'

In [21]:
val_dir = base_dir / 'validation'

In [22]:
base_dir

PosixPath('cats_and_dogs_filtered')

In [23]:
train_dir

PosixPath('cats_and_dogs_filtered/train')

In [24]:
val_dir

PosixPath('cats_and_dogs_filtered/validation')

In [25]:
train_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

Found 2000 files belonging to 2 classes.


In [28]:
val_ds = keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 1000 files belonging to 2 classes.


In [30]:
class_names = train_ds.class_names

In [31]:
class_names

['cats', 'dogs']

In [32]:
# 성능최적화를 위해서, 다음 배치를 백그라운드에서 미리 준비하라는 코드 작성

In [33]:
AUTOTUNE = tf.data.AUTOTUNE

In [34]:
train_ds = train_ds.prefetch(AUTOTUNE)

In [35]:
val_ds = val_ds.prefetch(AUTOTUNE)

In [36]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        layers.RandomTranslation(0.1, 0.1)
    ],
    name="data_augmentation",
)

In [38]:
def build_model(input_shape) :
  return keras.Sequential(
      [
          layers.Input(shape= input_shape ),
          data_augmentation,
          layers.Rescaling( 1.0 / 255.0 ),
          layers.Conv2D( 32, 3, activation='relu' ),
          layers.MaxPooling2D(2, 2),
          layers.Conv2D( 64, 3, activation='relu' ),
          layers.MaxPooling2D(2, 2),
          layers.Conv2D( 128, 3, activation='relu' ),
          layers.MaxPooling2D(2, 2),

          layers.Flatten(),
          layers.Dense( 512, activation='relu' ),
          layers.Dropout(0.2),
          layers.Dense( 1, activation='sigmoid' )
      ],
      name="simple_cnn",
  )